# 18 · Signed J-space transport (standalone Exp 7)

Runs **only** experiment 7 from notebook 16: per direction, transport *magnitude* (`‖J·v̂‖²`
gain) **and sign** (`cos(J·v̂, v̂)`) — does the verbal workspace transmit each component and
sub-trait faithfully or inverted? Directions: shared / dark-specific / depression-specific
components, dark sub-traits (incl. whole-instrument Machiavellianism), depression sub-traits
(rumination, hopelessness, worry, dysregulation, avoidance). Three lenses (base, dark,
clinical-depression), random-vector noise floor, joined against Exp 6's divergence JSON if
present in the (tagged) `components_v1` dir.

Needs: shift pickles (06c), battery rows (09, item lists only), lenses (10, HF or the Drive
`jacobian_lenses/` copies), and per-item activations for **base + dark + clinical-depression**
(extracted here if missing; shares 16/17's cache dirs).

**To run on the v1 organisms** (before meta_1's cleanup deletes their files): set
`RUN_TAG = "_v1"` and point the organisms at the v1 repos in the config cell — dark:
`Koalacrown/dark-qwen3-8b-rl-merged`, clinical-depression:
`Koalacrown/clinical-depression-qwen3-8b`. The 17 v1 run already cached dark's activations;
this adds base + clinical (~20 min on L4). The lens fetch falls back to Drive, where the v1
lenses still live.

## 1. Setup

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
%pip install -q -U "numpy>=2.1" "scipy>=1.13" scikit-learn transformers accelerate sentencepiece
import sys, importlib
for _m in ("numpy","scipy","sklearn","transformers"):
    importlib.import_module(_m); print(_m, "->", getattr(sys.modules[_m], "__version__", "ok"))

In [ ]:
import pathlib
DRIVE = mount_drive()
use_probe_repo()
RUN_TAG = ""      # "" = current organisms (shares item_acts_v1/components_v1 with 16/17).
                  # "_v1" = old organisms -> item_acts_v1_v1 / components_v1_v1 (17's v1 dirs).
DIRS  = (DRIVE / "directions_v1")             if DRIVE else pathlib.Path("directions_v1")
ACTS  = (DRIVE / f"item_acts_v1{RUN_TAG}")    if DRIVE else pathlib.Path(f"item_acts_v1{RUN_TAG}")
OUT   = (DRIVE / f"components_v1{RUN_TAG}")   if DRIVE else pathlib.Path(f"components_v1{RUN_TAG}")
for p in (ACTS, OUT): p.mkdir(parents=True, exist_ok=True)

if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass

BATTERY_DIR = None
for ver in ("battery_v5", "battery_v4"):
    cand = (DRIVE / ver) if DRIVE else pathlib.Path(ver)
    if (cand / "rows_dark.csv").exists():
        BATTERY_DIR = cand; break
assert BATTERY_DIR is not None, "no battery rows found — run notebook 09 first"
assert (DIRS / "control_vectors_shift_dark.pkl").exists(), "shift vectors missing — run 06c"
print("directions <-", DIRS, "| battery <-", BATTERY_DIR, "| acts ->", ACTS, "| out ->", OUT)

## 2. Config
All three organisms — the depression sub-trait directions need the clinical organism's (and
base's) activations. For the v1 run swap the `hf` ids (see header).

In [ ]:
ORGANISMS = [
    {"name": "base",                "hf": "Qwen/Qwen3-8B"},
    {"name": "dark",                "hf": "Koalacrown/dark-2-qwen3-8b"},                # v1: Koalacrown/dark-qwen3-8b-rl-merged
    {"name": "clinical-depression", "hf": "Koalacrown/clinical-2-qwen3-8b"},            # v1: Koalacrown/clinical-depression-qwen3-8b
]
ACT_LAYERS = list(range(16, 25)) + list(range(30, 35))
PROBE_L    = 18
SELECTOR   = "task_mean"
BATCH      = 16
MAXTOK     = 512
SKIP_EXISTING = True
BANDS      = {"mid (16-24)": range(16, 25), "late (30-34)": range(30, 35)}
print(f"{len(ORGANISMS)} organisms | layers {ACT_LAYERS} | selector {SELECTOR}")

## 3. Items + battery scores
Battery items from `data/source_items/*.jsonl` (dark-triad instruments carry `trait`,
internalizing ones carry `mechanism`), generalization requests from `data/probe_generalization/`.
Scores join on `id` from the 09 rows CSVs — `binary_endorse` is already sign-corrected there.

In [ ]:
import json, glob, csv, collections

def load_jsonl(p):
    return [json.loads(l) for l in open(p) if l.strip()]

ITEMS = {}                       # id -> item dict (+ "side": "trait"|"mechanism", "instrument")
for f in sorted(glob.glob("/content/dt_rl/data/source_items/*.jsonl")):
    inst = pathlib.Path(f).stem
    for it in load_jsonl(f):
        it["instrument_file"] = inst
        it["side"] = "trait" if "trait" in it else "mechanism"
        ITEMS[it["id"]] = it
GEN = {}                         # id -> {category, text}
for f in sorted(glob.glob("/content/dt_rl/data/probe_generalization/*.jsonl")):
    for it in load_jsonl(f):
        GEN[it["id"]] = it

ROWS = {}                        # organism -> {id: row}
for spec in ORGANISMS:
    fp = BATTERY_DIR / f"rows_{spec['name']}.csv"
    if fp.exists():
        ROWS[spec["name"]] = {r["id"]: r for r in csv.DictReader(open(fp))}
    else:
        print(f"!! rows_{spec['name']}.csv missing — Exp 1-3 will skip this organism")

# ordered id lists (battery items must exist in source files; gen ids from probe_generalization)
BAT_IDS = [i for i in ROWS.get("dark", ROWS.get("base", {})) if i in ITEMS]
GEN_IDS = [i for i in ROWS.get("dark", ROWS.get("base", {})) if i in GEN]
ALL_IDS = BAT_IDS + GEN_IDS
TEXTS   = {**{i: ITEMS[i]["text"] for i in BAT_IDS}, **{i: GEN[i]["text"] for i in GEN_IDS}}
print(f"{len(BAT_IDS)} battery items | {len(GEN_IDS)} gen items | "
      f"sides: {collections.Counter(ITEMS[i]['side'] for i in BAT_IDS)}")

## 4. Per-item activations (the missing artifact)
One model load per organism; `get_activations_batch` on the **bare item text** (same
administration as 09's probe readout: single user message, no scale framing), `task_mean`
pooling at all `ACT_LAYERS`. Saved to `item_acts_v1/acts_items_<name>.npz` (fp16, ~75 MB each).

In [ ]:
import numpy as np, torch, gc
from tqdm.auto import tqdm
from src.models.huggingface_model import HuggingFaceModel

def extract_org(spec):
    name = spec["name"]; fp = ACTS / f"acts_items_{name}.npz"
    if SKIP_EXISTING and fp.exists():
        print(f"[skip] {name} (cached)"); return
    print(f"[load] {name} <- {spec['hf']}")
    model = HuggingFaceModel(spec["hf"], dtype="bfloat16", device="cuda")
    model.tokenizer.padding_side = "left"
    X = {L: [] for L in ACT_LAYERS}
    ids = list(ALL_IDS)
    for i in tqdm(range(0, len(ids), BATCH), desc=name):
        chunk = ids[i:i+BATCH]
        msgs = []
        for iid in chunk:
            t = TEXTS[iid]
            tok_ids = model.tokenizer(t, add_special_tokens=False).input_ids
            if len(tok_ids) > MAXTOK:
                t = model.tokenizer.decode(tok_ids[:MAXTOK])
            msgs.append([{"role": "user", "content": t}])
        res = model.get_activations_batch(msgs, ACT_LAYERS, [SELECTOR])
        for L in ACT_LAYERS:
            X[L].append(np.asarray(res[SELECTOR][L], dtype=np.float16))
    np.savez_compressed(fp, ids=np.array(ids),
                        **{f"L{L}": np.concatenate(X[L]) for L in ACT_LAYERS})
    print(f"[done] {name} -> {fp.name}")
    del model; gc.collect(); torch.cuda.empty_cache()

for spec in ORGANISMS:
    extract_org(spec)

def load_acts(name):
    z = np.load(ACTS / f"acts_items_{name}.npz")
    ids = list(z["ids"])
    idx = {i: j for j, i in enumerate(ids)}
    return {L: z[f"L{L}"].astype(np.float32) for L in ACT_LAYERS}, idx

ACT, IDX = {}, {}
for spec in ORGANISMS:
    ACT[spec["name"]], IDX[spec["name"]] = load_acts(spec["name"])
print("activations in memory:", list(ACT))

## 5. Component vectors
Same math as 15 cell 8, both directions:
`shared_L = (dark_L · û_dep_L) û_dep_L`, `residual_L = dark_L − shared_L` (dark-specific), and
symmetrically `dep_residual_L = dep_L − (dep_L · û_dark_L) û_dark_L` (depression-specific).

In [ ]:
import pickle

def load_shift(name):
    return pickle.load(open(DIRS / f"control_vectors_shift_{name}.pkl", "rb"))["vectors"]["induced_shift"]

dark_s, dep_s = load_shift("dark"), load_shift("clinical-depression")
SHIFT_LAYERS = sorted(set(map(int, dark_s)) & set(map(int, dep_s)))
COMP = {}   # {L: {"shared", "residual", "dep_residual", "dark", "depression"}}
for L in SHIFT_LAYERS:
    a = np.asarray(dark_s[L], np.float32); b = np.asarray(dep_s[L], np.float32)
    u_dep, u_dark = b / np.linalg.norm(b), a / np.linalg.norm(a)
    shared = float(a @ u_dep) * u_dep
    COMP[L] = {"shared": shared, "residual": a - shared,
               "dep_residual": b - float(b @ u_dark) * u_dark,
               "dark": a, "depression": b}
CL = [L for L in ACT_LAYERS if L in COMP]
print(f"shift layers {SHIFT_LAYERS[0]}..{SHIFT_LAYERS[-1]} | usable with acts: {CL}")

## 6. Exp 7 — signed J-space transport (magnitude × sign)
Exp 4 measured how much of each direction the verbal workspace transports (`‖J·v̂‖²` gain).
This measures **which way**: `cos(J·v̂, v̂)` — does the part that gets through point the same
way (faithful verbalization) or the opposite way (inversion)? `J_l` maps layer-l residual to
final-layer residual (`[d_model, d_model]`, jlens), so input and output live in the same stream.

Directions: the three components (shared / dark-specific / depression-specific), dark sub-traits
incl. **whole-instrument Machiavellianism** (mach_iv, absent from Exp 4), and depression
sub-traits (rumination, hopelessness, worry, dysregulation, avoidance) built symmetrically from
the depression organism's acts. No anhedonia instrument exists in the battery — not proxied.

Predictions: shared & depression-specific & all depression sub-traits → decent gain, **positive**
cos. Overt dark (admiration, boldness) → positive cos. Covert dark (Machiavellianism,
disinhibition… the Exp 6 "probe high / binary low" tail) → low gain and **negative/zero** cos.
Guardrails: the 64-random-vector null gives the cos noise floor (a "negative" cos must clear it);
`ref_shared` is the positive control — if even it sits at cos ≈ 0, the sign readout is
uninformative and we say so. J is a wikitext-averaged linearization; sign claims are about the
average pathway. Joined against Exp 6's divergence if its JSON is present.

In [ ]:
import torch
from huggingface_hub import hf_hub_download
DEV = "cuda" if torch.cuda.is_available() else "cpu"

#              name              organism                inst       subscale         prediction
DIRSPEC = {
    "machiavellianism": ("dark",                "mach_iv", None,            "covert -> low gain, -cos"),
    "sd3_mach":         ("dark",                "sd3",     "Machiavellianism","covert -> low gain, -cos"),
    "disinhibition":    ("dark",                "tripm",   "disinhibition",  "covert-ish -> -/0 cos"),
    "rivalry":          ("dark",                "narq",    "rivalry",        "mixed"),
    "meanness":         ("dark",                "tripm",   "meanness",       "mixed"),
    "boldness":         ("dark",                "tripm",   "boldness",       "overt -> +cos"),
    "admiration":       ("dark",                "narq",    "admiration",     "overt -> +cos"),
    "npi_grandiosity":  ("dark",                "npi40",   None,             "overt -> +cos"),
    "rumination_brood": ("clinical-depression", "rrs",     "brooding",       "dystonic -> +cos"),
    "rumination_dep":   ("clinical-depression", "rrs",     "depression",     "dystonic -> +cos"),
    "hopelessness":     ("clinical-depression", "bhs",     None,             "dystonic -> +cos"),
    "worry":            ("clinical-depression", "pswq",    None,             "dystonic -> +cos"),
    "dysregulation":    ("clinical-depression", "ders16",  None,             "dystonic -> +cos"),
    "avoidance":        ("clinical-depression", "aaq2",    None,             "dystonic -> +cos"),
}

def spec_ids(inst, sub):
    return [i for i in BAT_IDS
            if ITEMS[i]["instrument_file"] == inst
            and (sub is None or ITEMS[i].get("subscale") == sub)
            and not ITEMS[i].get("reverse_keyed", False)]

SPEC_IDS = {n: spec_ids(inst, sub) for n, (org, inst, sub, _) in DIRSPEC.items()}
for n, ids in SPEC_IDS.items():
    print(f"  {n:17s} {len(ids):2d} items  ({DIRSPEC[n][0]}, {DIRSPEC[n][1]}/{DIRSPEC[n][2]})")

def dmean(org, L, ids):
    return ACT[org][L][[IDX[org][i] for i in ids]].mean(0)

DIR7 = {L: {} for L in ACT_LAYERS}
for L in ACT_LAYERS:
    for n, (org, inst, sub, _) in DIRSPEC.items():
        ids = SPEC_IDS[n]
        if len(ids) >= 4 and org in ACT:
            DIR7[L][n] = dmean(org, L, ids) - dmean("base", L, ids)
    if L in COMP:
        for c in ("shared", "residual", "dep_residual"):
            DIR7[L][f"ref_{c}"] = COMP[L][c]
print("directions per layer:", len(DIR7[ACT_LAYERS[0]]))

LENSES7 = {
    "base": ("neuronpedia/jacobian-lens",
             "qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt"),
    "dark": ("Koalacrown/jacobian-lens-organisms", "dark/jacobian_lens.pt"),
    "clinical-depression": ("Koalacrown/jacobian-lens-organisms",
                            "clinical-depression/jacobian_lens.pt"),
}

def load_J(lname, repo, fname):
    try:
        path = hf_hub_download(repo, fname, token=os.environ.get("HF_TOKEN") or None)
    except Exception as e:
        local = DRIVE / "jacobian_lenses" / f"{lname}_jacobian_lens.pt"
        assert local.exists(), f"lens {lname}: HF failed ({type(e).__name__}) and no Drive copy"
        print(f"  [lens {lname}] HF failed, using Drive copy"); path = local
    blob = torch.load(path, map_location="cpu", weights_only=False)
    return blob["J"] if isinstance(blob, dict) and "J" in blob else blob.jacobians

rng7 = np.random.default_rng(0)
RAND7 = rng7.standard_normal((64, 4096)).astype(np.float32)
RAND7 /= np.linalg.norm(RAND7, axis=1, keepdims=True)
R_t = torch.tensor(RAND7)

SIGNED = {}   # {lens: {L: {name: {"gain_rel","cos","cos_z"} , "_null": {...}}}}
for lname, (repo, fname) in LENSES7.items():
    J_all = load_J(lname, repo, fname)
    layers = [L for L in ACT_LAYERS if L in J_all]
    print(f"\n== lens {lname}: layers {layers[0]}..{layers[-1]} ==")
    SIGNED[lname] = {}
    for L in layers:
        J = J_all[L].float().to(DEV)
        names = list(DIR7[L])
        V = np.stack([DIR7[L][n] / np.linalg.norm(DIR7[L][n]) for n in names])
        JV = (J @ torch.tensor(V).to(DEV).T).T.cpu().numpy()
        JR = (J @ R_t.to(DEV).T).T.cpu().numpy()
        g_null = (JR ** 2).sum(1)
        c_null = (JR * RAND7).sum(1) / (np.linalg.norm(JR, axis=1) + 1e-12)
        gm, cm, cs = float(g_null.mean()), float(c_null.mean()), float(c_null.std() + 1e-12)
        SIGNED[lname][L] = {"_null": {"cos_mean": cm, "cos_sd": cs}}
        for k, n in enumerate(names):
            g = float((JV[k] ** 2).sum())
            c = float(JV[k] @ V[k] / (np.linalg.norm(JV[k]) + 1e-12))
            SIGNED[lname][L][n] = {"gain_rel": g / gm, "cos": c, "cos_z": (c - cm) / cs}
        del J
        if DEV == "cuda": torch.cuda.empty_cache()
    del J_all

# band summary + optional join with Exp 6 divergence
div_by_key = {}
_e6 = OUT / "exp6_probe_binary_divergence.json"
if _e6.exists():
    for grp in json.load(open(_e6))["groups"]:
        div_by_key[(grp["instrument"], grp["subscale"].lower())] = grp["mean_div"]

def dir_div(n):
    if n not in DIRSPEC: return None
    _, inst, sub, _ = DIRSPEC[n]
    hits = [v for (gi, gs), v in div_by_key.items()
            if gi == inst and (sub is None or gs == sub.lower())]
    return float(np.mean(hits)) if hits else None

GROUPS7 = [("components", ["ref_shared", "ref_dep_residual", "ref_residual"]),
           ("dark sub-traits", [n for n, s in DIRSPEC.items() if s[0] == "dark"]),
           ("depression sub-traits", [n for n, s in DIRSPEC.items() if s[0] != "dark"])]
EXP7 = []
for lname, per_layer in SIGNED.items():
    print(f"\n===== lens: {lname} =====")
    for bname, rng_ in BANDS.items():
        Ls = [L for L in per_layer if L in rng_]
        if not Ls: continue
        csd = np.mean([per_layer[L]["_null"]["cos_sd"] for L in Ls])
        print(f"\n  {bname}  (random-dir cos noise sd = {csd:.3f})")
        print(f"  {'direction':17s} {'prediction':26s} {'gain':>7s} {'cos':>8s} {'cos_z':>7s} {'exp6_div':>9s}")
        for gname, ns in GROUPS7:
            avail = [n for n in ns if n in per_layer[Ls[0]]]
            if not avail: continue
            print(f"  -- {gname}")
            rows = []
            for n in avail:
                gain = np.mean([per_layer[L][n]["gain_rel"] for L in Ls if n in per_layer[L]])
                cos  = np.mean([per_layer[L][n]["cos"]      for L in Ls if n in per_layer[L]])
                cz   = np.mean([per_layer[L][n]["cos_z"]    for L in Ls if n in per_layer[L]])
                rows.append((n, gain, cos, cz, dir_div(n)))
            for n, gain, cos, cz, dv in sorted(rows, key=lambda r: -r[2]):
                pred = DIRSPEC[n][3] if n in DIRSPEC else "reference"
                dvs = f"{dv:+9.2f}" if dv is not None else "        -"
                print(f"  {n:17s} {pred:26s} {gain:6.2f}x {cos:+8.3f} {cz:+7.1f} {dvs}")
                EXP7.append({"lens": lname, "band": bname, "direction": n, "prediction": pred,
                             "gain_rel": float(gain), "cos": float(cos), "cos_z": float(cz),
                             "exp6_div": dv})
        both = [(r["cos"], r["exp6_div"]) for r in EXP7
                if r["lens"] == lname and r["band"] == bname
                and r["exp6_div"] is not None and r["direction"] in DIRSPEC
                and DIRSPEC[r["direction"]][0] == "dark"]
        if len(both) >= 4:
            from scipy.stats import spearmanr
            rho = spearmanr([b[0] for b in both], [b[1] for b in both])[0]
            print(f"  spearman(cos, exp6 divergence) over {len(both)} dark sub-traits: {rho:+.2f}")

with open(OUT / "exp7_signed_transport.json", "w") as f:
    json.dump(EXP7, f, indent=2)
print("\nsaved ->", OUT / "exp7_signed_transport.json")

---
# Done
`exp7_signed_transport.json` in the (tagged) `components_v1` dir. The paper claim this feeds:
magnitude gates *how much* reaches verbal output, sign decides whether it arrives *faithful or
inverted* — high gain + positive cos = reported (depression), low gain + negative cos = denied
(covert dark), high gain + positive cos on overt dark sub-traits = performed.